# Lab 1：静态页面与 API 数据采集

**姓名：** 刘嘉  
**学号：** 112404260113  
**班级：** 数据科学与大数据技术1241班

本 Notebook 完成两类典型的 Web 数据采集：

```text
Device ：动态加载 → 找到 API → JSON → 分页采集全部设备（5000 条）
Company：静态加载 → HTML → BeautifulSoup → "下一页" → 采集 3 页（60 条）
```

目标平台：SmartPark DataLab（`https://115.159.159.82`）


## 0. 准备工作

导入依赖并设置公共变量。

In [1]:
import json
import warnings
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# 实验平台为教学用自签名证书，关闭 HTTPS 验证警告
warnings.filterwarnings("ignore")

BASE_URL = "https://115.159.159.82"
API_URL = BASE_URL + "/api/devices"

print("requests:", requests.__version__)
print("BeautifulSoup 已导入")


requests: 2.34.2
BeautifulSoup 已导入


---
# 任务 1：判断数据加载方式

## 1.1 Device 页面：判断为动态加载

**判断依据：**

1. 在 `/devices` 页面使用「右键 → 查看网页源代码」，按 `Ctrl + F` 搜索页面上真实显示的设备名称（例如 `循环泵-000001-001`），**搜不到**。
   说明设备记录不随初始 HTML 一起返回。
2. 打开 `F12 → Network → Fetch/XHR` 并刷新页面，可以找到返回设备列表的请求：

```text
Request URL   : https://115.159.159.82/api/devices?page=1&page_size=50
Request Method: GET
Query 参数     : page=1, page_size=50
```

因此 Device 页面属于**动态加载**，数据需要通过 API 获取。

> 证据截图见 `images/01_device_api.png`（浏览器 DevTools 的 Network 面板）。


## 1.2 复现第一页请求

用 `requests` 复现 Network 中观察到的第一个请求。

In [2]:
r = requests.get(
    API_URL,
    params={"page": 1, "page_size": 50},
    timeout=10,
    verify=False,
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))


status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


读取 JSON 并观察顶层结构。

In [3]:
data = r.json()

print("type:", type(data))
print("keys:", data.keys())


type: <class 'dict'>
keys: dict_keys(['page', 'page_size', 'total', 'items'])


In [4]:
# 查看完整结构（截断显示，避免刷屏）
print(json.dumps({k: v for k, v in data.items() if k != "items"},
                 ensure_ascii=False, indent=2))
print("items 条数:", len(data["items"]))


{
  "page": 1,
  "page_size": 50,
  "total": 5000
}
items 条数: 50


查看一条真实设备记录，确认各字段含义。

In [5]:
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))


{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


## 1.3 分页结构

从第一页 JSON 中可以读出分页相关的 4 个字段：

| 含义     | JSON 字段     | 本次取值 |
| ------ | ----------- | ---: |
| 当前页    | `page`      |    1 |
| 每页条数   | `page_size` |   50 |
| 设备总数   | `total`     | 5000 |
| 本页设备列表 | `items`     |   50 |


---
# 任务 2：Device API 分页采集

根据 `total` 和 `page_size` **计算**总页数，不写死页数。

In [6]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)


total: 5000
page_size: 50
total_pages: 100


循环采集全部页面。

In [7]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        API_URL,
        params={"page": page, "page_size": page_size},
        timeout=10,
        verify=False,
    )
    page_data = r.json()
    page_items = page_data["items"]
    devices.extend(page_items)

    if page % 20 == 0 or page == total_pages:
        print(f"已采集 {page:>3}/{total_pages} 页，累计 {len(devices)} 条")

print()
print("device count:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))


已采集  20/100 页，累计 1000 条


已采集  40/100 页，累计 2000 条


已采集  60/100 页，累计 3000 条


已采集  80/100 页，累计 4000 条


已采集 100/100 页，累计 5000 条

device count: 5000
unique device_id: 5000


---
# 任务 3：Company 页面：判断为静态加载

**判断依据：** 在 `/companies` 页面使用「右键 → 查看网页源代码」，`Ctrl + F` 搜索页面上真实显示的企业名称（例如 `临港精工001有限公司`），**能够直接搜到**。
说明企业记录随初始 HTML 一起返回，属于**静态加载**，可以直接用 `requests` + `BeautifulSoup` 解析。

> 证据截图见 `images/02_company_source.png`（网页源代码页面）。

## 3.1 定位一条企业记录

先请求 HTML 并转为 `BeautifulSoup` 树形结构。

In [8]:
current_url = BASE_URL + "/companies"

r = requests.get(current_url, timeout=10, verify=False)
soup = BeautifulSoup(r.text, "html.parser")

print("status:", r.status_code)
print("html 长度:", len(r.text))


status: 200
html 长度: 11331


在 Elements 中观察一条完整企业记录最外层的节点，它的结构是：

```html
<article class="company-card" data-company-id="C000001">
  <div class="company-card-top">
    <span class="mono">C000001</span>
    <span class="badge active">运营中</span>
  </div>
  <h2><a href="/companies/C000001">临港精工001有限公司</a></h2>
  <dl>
    <div><dt>所属行业</dt><dd>汽车零部件</dd></div>
    <div><dt>所在区域</dt><dd>智造一区</dd></div>
    <div><dt>员工规模</dt><dd>88 人</dd></div>
    <div><dt>设备数量</dt><dd>6 台</dd></div>
  </dl>
  <a class="detail-link" href="/companies/C000001">查看企业档案 →</a>
</article>
```

**Selector 设计思路**（按优先级逐条考察）：

| 优先级 | 观察 | 结论 |
| --- | --- | --- |
| ① id | 记录节点没有 `id` | 排除 |
| ② class | 有语义明确的 `class="company-card"`，且每条记录都有 | **采用** |
| ③ data-* | 有 `data-company-id`，可用来取企业编号，但不适合作为"选出全部记录"的选择器 | 用它取字段 |
| ④ 父子层级 | 无需依赖 | 不用 |
| ⑤ 位置关系 | 无需依赖 | 不用 |

最终采用 **`article.company-card`**：短、语义明确，只依赖标签名与稳定 class，对页面其他结构调整不敏感。

> 证据截图见 `images/03_company_next.png`。


In [9]:
cards = soup.select("article.company-card")

print("本页企业数:", len(cards))


本页企业数: 20


## 3.2 提取企业字段

先解析第一条记录，确认每个字段的取法。

In [10]:
card = cards[0]

# company_id 来自 data-* 属性
company_id = card.get("data-company-id")

# name 位于记录内部的 h2 > a 文本
name = card.select_one("h2 a").get_text(strip=True)

# status 位于记录内部的 span.badge 文本
status = card.select_one("span.badge").get_text(strip=True)

print("company_id:", company_id)
print("name:", name)
print("status:", status)


company_id: C000001
name: 临港精工001有限公司
status: 运营中


行业、区域、设备数量采用 `dt/dd` 结构，先转换成字典。

In [11]:
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)


{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [12]:
industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print("industry:", industry)
print("zone:", zone)
print("device_count:", device_count, type(device_count))


industry: 汽车零部件
zone: 智造一区
device_count: 6 <class 'int'>


### 完成第一页循环

按手册 4.2 要求，把上面单条记录的取法套到**第 1 页的全部 20 条记录**上，得到 `companies` 列表。

In [13]:
companies = []

for card in cards:
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    companies.append({
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one("span.badge").get_text(strip=True),
    })

print("本页企业数:", len(companies))

本页企业数: 20


### 字段来源说明

1. **`company_id`**
   - 来自：HTML 属性
   - 使用的方法：`card.get("data-company-id")`，直接从记录节点的 `data-*` 自定义属性读取，比从文本里截取更稳定。
2. **`device_count`**
   - 来自：文本（`dt/dd` 结构）
   - 使用的方法：先用 `card.select("dl > div")` 遍历，把 `dt`/`dd` 组装成字典，再取 `details["设备数量"]`，最后用 `int()` 把 `"6 台"` 转成整数 `6`。

## 3.3 定位"下一页"并补全 URL

先在 Elements 中观察页面底部的"下一页"链接，设计一个稳定的 selector 并验证。

In [14]:
# 设计一个稳定的 selector，并验证
next_link = soup.select_one("nav.pagination a:last-child")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))


下一页
/companies?page=2


In [15]:
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)


https://115.159.159.82/companies?page=2


---
# 任务 4：根据"下一页"连续采集 3 页

**"下一页" 元素的观察结果：**

```html
<nav class="pagination" aria-label="企业分页">
  <span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a>
</nav>
```

- 标签：`<a>`
- 所在容器：`nav.pagination`
- `href`：`/companies?page=2`（**相对路径**，需要用 `urljoin` 补全）

**Selector 设计：** 先看整体结构——`nav.pagination` 里第 1 页时是 `上一页(span)` + `1(b)` + `下一页(a)`，
只有一个 `<a>`，所以 `nav.pagination a` 就能选中它。这里进一步写成 `nav.pagination a:last-child`，
用 `:last-child` 明确指向"导航里的最后一个链接"，语义更直接。

为了更稳，代码里再按**链接文本包含"下一页"**做一次过滤——这样即使将来前面新增其他 `<a>` 也不会选错。

用 `urljoin` 把相对路径补成完整 URL。

把下面三个步骤组合到循环中，连续采集 3 页：

```text
① 请求当前页
② 解析本页企业
③ 找到下一页 href
```

关键点：**③ 的输出（下一页 URL）就是下一轮 ① 的输入**，循环靠这个变量接力向前推进。

In [16]:
# 单独验证"下一页"元素的取法
next_link = next(
    (a for a in soup.select("nav.pagination a") if "下一页" in a.get_text()),
    None,
)

print("文本:", next_link.get_text(strip=True))
print("href:", next_link.get("href"))

next_url = urljoin(current_url, next_link.get("href"))
print("完整 URL:", next_url)


文本: 下一页
href: /companies?page=2
完整 URL: https://115.159.159.82/companies?page=2


In [17]:
companies = []
current_url = BASE_URL + "/companies"    # 从第 1 页出发

page_stats = []

for page_no in range(1, 4):              # 最多采 3 页

    # ① 请求当前页
    r = requests.get(current_url, timeout=10, verify=False)
    soup = BeautifulSoup(r.text, "html.parser")

    # ② 解析本页企业
    rows = []
    for card in soup.select("article.company-card"):
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = \
                item.select_one("dd").get_text(strip=True)

        rows.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台", "").strip()),
            "status": card.select_one("span.badge").get_text(strip=True),
        })

    companies.extend(rows)
    page_stats.append((page_no, len(rows)))

    # ③ 找到下一页 href，补全后作为下一轮的"当前页"
    next_link = next(
        (a for a in soup.select("nav.pagination a") if "下一页" in a.get_text()),
        None,
    )
    if next_link is None:
        print(f"第 {page_no} 页：{len(rows)} 条  |  已是最后一页")
        break

    current_url = urljoin(current_url, next_link.get("href"))
    print(f"第 {page_no} 页：{len(rows)} 条  |  下一页 -> {current_url}")

print()
print("Company 总数:", len(companies))

第 1 页：20 条  |  下一页 -> https://115.159.159.82/companies?page=2
第 2 页：20 条  |  下一页 -> https://115.159.159.82/companies?page=3


第 3 页：20 条  |  下一页 -> https://115.159.159.82/companies?page=4

Company 总数: 60


**循环逻辑说明：** 程序先请求当前页并解析出本页企业记录，同时从页面底部取出"下一页"的 `href`；
用 `urljoin` 补成完整 URL 后作为下一轮的请求地址，如此重复。当找不到"下一页"时（返回 `None`）就结束循环。
本实验固定采集 3 页，因此 `range(1, 4)` 同时提供了次数上限，避免死循环。

## 4.1 三页采集结果

| 页码 | 企业数 |
| --- | --: |

In [18]:
print(f"{'页码':<6}{'企业数':>8}")
print("-" * 14)
for page_no, cnt in page_stats:
    print(f"{page_no:<6}{cnt:>8}")
print("-" * 14)
print(f"{'合计':<6}{len(companies):>8}")

print()
print("唯一 company_id:", len(set(x["company_id"] for x in companies)))


页码         企业数
--------------
1           20
2           20
3           20
--------------
合计          60

唯一 company_id: 60


---
# 任务 5：结果验证

## 5.1 数量与 ID 唯一性

In [19]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))
print()
print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))


companies: 60
unique company_id: 60

devices: 5000
unique device_id: 5000


## 5.2 Company–Device 一致性检查

用全部 Device 的 `company_id` 统计每家企业实际设备数，再与 Company 页面给出的 `device_count` 比较。

In [20]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)

    if expected != actual:
        mismatch_count += 1

print("不一致企业数:", mismatch_count)


不一致企业数: 0


---
# 任务 6：保存采集结果

In [21]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print(" -", data_dir / "devices.json", f"({len(devices)} 条)")
print(" -", data_dir / "companies.json", f"({len(companies)} 条)")


数据已保存到 data/ 目录
 - data/devices.json (5000 条)
 - data/companies.json (60 条)


---
# 最终验证结果

> 证据截图见 `images/04_validation.png`（JupyterLab 运行结果）。

In [22]:
print("=" * 46)
print("  Lab1 最终验证结果")
print("=" * 46)
print("Company 总数                :", len(companies))
print("唯一 company_id             :", len(set(x["company_id"] for x in companies)))
print("Device 总数                 :", len(devices))
print("唯一 device_id              :", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数  :", mismatch_count)
print("=" * 46)


  Lab1 最终验证结果
Company 总数                : 60
唯一 company_id             : 60
Device 总数                 : 5000
唯一 device_id              : 5000
Company-Device 不一致企业数  : 0


---
# 实验结论

本次实验对比了两类典型页面的采集方式。

**静态加载 vs 动态加载：** Company 页面的数据随初始 HTML 一起返回，因此可以直接用 `requests` 获取 HTML 再用
BeautifulSoup 解析；Device 页面在源代码中找不到设备记录，数据由前端异步请求 API 后渲染，必须先通过 Network
面板定位接口，再直接请求 JSON。

**两种分页的差异：** API 分页由 `page` / `page_size` 参数控制，总页数可以从响应里的 `total` 计算得到，
不需要依赖页面结构；HTML 分页则要靠解析页面上的"下一页"`<a href>`，再用 `urljoin` 补全相对路径后请求下一页。

**结果验证：** 我检查了采集数量（Company 60 条、Device 5000 条）与 ID 唯一性，并用 Device 的 `company_id`
分组统计与 Company 页面的 `device_count` 逐一比对，不一致企业数为 0，说明两个数据源的采集结果相互吻合。
